# 07 — Selective Agent Experiments

**Reconstructed 2026-09-25 from already-saved results (`data/agent_experiment.json`,
`results/runs/run_T041_final_test_rag*.jsonl`).** No new model or API calls were made to build
this notebook — every number below is loaded directly from files already written by
`scripts/run_agent_experiment.py` (dev-sample run) and `scripts/run_final_test_evaluation.py` /
`scripts/run_hosted_comparison.py` (T041 final-test run).

## 1. Research question

Does selective agentic investigation improve over the frozen RAG baseline on cases the
confidence/routing signal flags as REVIEW, without introducing an unacceptable rate of
regressions (a correct RAG answer flipped to wrong)?

## 2. Inputs / reused artefacts

- **Base cases:** the 150-case stratified dev sample (seed=42) used throughout this project's
  development phase, and separately the 500-case stratified T041 test-set subsample.
- **Baseline predictions:** RAG (retrieve → rerank → rule-boost → classify), no agent.
- **Agent predictions:** the same RAG context, escalated to the selective agent when
  `pipeline/confidence.py`'s `route()` returns REVIEW.
- **Routing criterion:** rule-baseline agreement with a plain (non-rule-boosted) RAG
  classification — see `docs/decisions.md` ADR-005/ADR-006 for why this signal was chosen and how
  its circularity was found and fixed.
- **Tool configuration:** 5 tools (`pipeline/agent_tools.py`): `search_clauses`,
  `find_defined_term`, `search_exceptions`, `retrieve_more_evidence`,
  `inspect_neighbouring_clauses`, in a bounded ReAct-style loop (`pipeline/agent.py`).
- **Saved output files:** `data/agent_experiment.json` (67-case dev-sample REVIEW subset, per-case
  outcomes), `results/runs/run_T041_final_test_rag_google_gemini-2.5-flash-lite.jsonl` and
  `..._rag_agent_google_gemini-2.5-flash-lite.jsonl` (500/2091-case hosted test-set predictions).

In [1]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

with open(REPO_ROOT / "data" / "agent_experiment.json") as f:
    dev = json.load(f)

print("Dev-sample REVIEW-subset agent experiment — top-level summary:")
for k, v in dev.items():
    if k != "outcomes":
        print(f"  {k}: {v}")
print(f"  n outcomes recorded: {len(dev['outcomes'])}")

Dev-sample REVIEW-subset agent experiment — top-level summary:
  n_review_cases: 67
  rag_accuracy_on_review: 0.8059701492537313
  agent_accuracy_on_review: 0.8507462686567164
  n_recovery: 6
  n_regression: 3
  n_no_change: 58
  avg_steps: 1.3134328358208955
  stopped_reasons: {'concluded': 52, 'duplicate_loop': 14, 'invalid_action': 1}
  total_cost_usd: 0.016855400000000003
  overall_rag_correct: 132
  overall_with_agent: 135
  n_total_sample: 150
  n outcomes recorded: 67


## 3. Architecture

Every REVIEW-routed case in this experiment already went through the full production retrieval
pipeline (sentence chunking → mpnet embeddings → retrieve top-20 → rerank with
`ms-marco-MiniLM-L-12-v2` → keep top-7 → RRF-fuse the rule match when it fires — `docs/decisions.md`
ADR-002) and an initial RAG classification. The agent is only invoked for the subset the routing
signal flags:

```text
RAG classification (rule-boosted context)
        |
Plain (non-rule-boosted) classification  --\
Rule-baseline label                      --> route(): ACCEPT or REVIEW
        |
   REVIEW only
        |
Selective agent: bounded ReAct loop over 5 tools,
duplicate-call detection, step/time/token limits
        |
Final label (agent's conclusion, or RAG's if the agent
never overrides it)
```

See `docs/architecture.md` for the full current request flow, including why two classifier calls
happen even before the agent is considered (the routing-independence fix, ADR-006).

## 4. Results

### 4a. Dev-sample REVIEW subset (67 cases, the original agent experiment)

In [2]:
print(f"REVIEW-routed cases: {dev['n_review_cases']} of {dev['n_total_sample']} total dev-sample cases "
      f"({dev['n_review_cases']/dev['n_total_sample']:.1%})")
print()
print(f"RAG accuracy on REVIEW subset:   {dev['rag_accuracy_on_review']:.1%}")
print(f"Agent accuracy on REVIEW subset: {dev['agent_accuracy_on_review']:.1%}")
print()
print(f"Recovery (RAG wrong -> agent right):   {dev['n_recovery']}/{dev['n_review_cases']}")
print(f"Regression (RAG right -> agent wrong): {dev['n_regression']}/{dev['n_review_cases']}")
print(f"No change:                             {dev['n_no_change']}/{dev['n_review_cases']}")
print()
print(f"Overall accuracy across full 150-case sample:")
print(f"  RAG only:        {dev['overall_rag_correct']}/{dev['n_total_sample']} = {dev['overall_rag_correct']/dev['n_total_sample']:.1%}")
print(f"  RAG + agent:     {dev['overall_with_agent']}/{dev['n_total_sample']} = {dev['overall_with_agent']/dev['n_total_sample']:.1%}")
print()
print(f"Avg agent steps/case: {dev['avg_steps']:.2f}")
print(f"Stop reasons: {dev['stopped_reasons']}")
print(f"Total agent cost (67 cases): ${dev['total_cost_usd']:.4f}")

REVIEW-routed cases: 67 of 150 total dev-sample cases (44.7%)

RAG accuracy on REVIEW subset:   80.6%
Agent accuracy on REVIEW subset: 85.1%

Recovery (RAG wrong -> agent right):   6/67
Regression (RAG right -> agent wrong): 3/67
No change:                             58/67

Overall accuracy across full 150-case sample:
  RAG only:        132/150 = 88.0%
  RAG + agent:     135/150 = 90.0%

Avg agent steps/case: 1.31
Stop reasons: {'concluded': 52, 'duplicate_loop': 14, 'invalid_action': 1}
Total agent cost (67 cases): $0.0169


### 4b. T041 final test-set run (much larger, real sample — hosted Gemini)

In [3]:
def load_latest(path):
    lines = [l for l in Path(path).read_text().splitlines() if l.strip()]
    return json.loads(lines[-1])

rag_path = REPO_ROOT / "results" / "runs" / "run_T041_final_test_rag_google_gemini-2.5-flash-lite.jsonl"
rag_agent_path = REPO_ROOT / "results" / "runs" / "run_T041_final_test_rag_agent_google_gemini-2.5-flash-lite.jsonl"

rag_rec = load_latest(rag_path)
rag_agent_rec = load_latest(rag_agent_path)

print(f"RAG record:       sample_size={rag_rec['config'].get('sample_size')}, "
      f"accuracy={rag_rec['metrics']['accuracy']:.3f}")
print(f"RAG+agent record: sample_size={rag_agent_rec['config'].get('sample_size')}, "
      f"accuracy={rag_agent_rec['metrics']['accuracy']:.3f}")

n_agent_used = sum(1 for p in rag_agent_rec['predictions'] if p.get('agent_used'))
n_total = len(rag_agent_rec['predictions'])
print(f"\nAgent routing rate: {n_agent_used}/{n_total} = {n_agent_used/n_total:.1%}")

# NOTE on the joint label+evidence metric: this project found (2026-09-24) that
# run_final_test_evaluation.py never populated retrieved_span_indices for any
# architecture, silently breaking the joint metric for the ENTIRE T041 run.
# As of this notebook's construction, only the hosted full_context architecture's
# joint value has been backfilled and verified correct - the rag/rag_agent joint
# values in these files are NOT yet corrected. See docs/evaluation_protocol.md's
# "Current evaluation status" section. Accuracy (used above) is unaffected by this bug.
print("\n(See docs/evaluation_protocol.md - the joint label+evidence metric in these files")
print(" is not yet backfilled for rag/rag_agent; accuracy is unaffected and used here.)")

rag_rec_accuracy = rag_rec["metrics"]["accuracy"]
agent_rec_accuracy = rag_agent_rec["metrics"]["accuracy"]


RAG record:       sample_size=2091, accuracy=0.787
RAG+agent record: sample_size=2091, accuracy=0.777

Agent routing rate: 900/2091 = 43.0%

(See docs/evaluation_protocol.md - the joint label+evidence metric in these files
 is not yet backfilled for rag/rag_agent; accuracy is unaffected and used here.)


## 5. Paired error analysis (dev-sample 67-case REVIEW subset)

Classifying every REVIEW-routed case into one of four buckets by comparing the RAG-only label and
the RAG+agent label against gold.

In [4]:
buckets = {"both_correct": 0, "both_wrong": 0, "rag_only_correct": 0, "agent_only_correct": 0}
for o in dev["outcomes"]:
    if o["rag_correct"] and o["agent_correct"]:
        buckets["both_correct"] += 1
    elif not o["rag_correct"] and not o["agent_correct"]:
        buckets["both_wrong"] += 1
    elif o["rag_correct"] and not o["agent_correct"]:
        buckets["rag_only_correct"] += 1  # regression
    else:
        buckets["agent_only_correct"] += 1  # recovery

for k, v in buckets.items():
    print(f"  {k}: {v}")

assert buckets["rag_only_correct"] == dev["n_regression"]
assert buckets["agent_only_correct"] == dev["n_recovery"]
print("\n(Cross-checked against the file's own n_recovery/n_regression fields - matches.)")

  both_correct: 51
  both_wrong: 7
  rag_only_correct: 3
  agent_only_correct: 6

(Cross-checked against the file's own n_recovery/n_regression fields - matches.)


## 6. Statistical test

McNemar's exact binomial test on the discordant pairs (b = RAG-only-correct/regression,
c = agent-only-correct/recovery), the same method already used and recorded in
`docs/decisions.md` ADR-007 — reproduced here directly from the saved per-case outcomes rather than
re-derived independently, so this is a verification, not a new analysis.

In [5]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from scipy import stats
from pipeline.parser import parse_contractnli_file
from pipeline.config import settings

ds = parse_contractnli_file(REPO_ROOT / "data" / "contractnli" / "test.json")
rag_pred_map = {(p["doc_id"], p["hypothesis_id"]): p["predicted_label"] for p in rag_rec["predictions"]}
agent_pred_map = {(p["doc_id"], p["hypothesis_id"]): p["predicted_label"] for p in rag_agent_rec["predictions"]}
keys = set(rag_pred_map) & set(agent_pred_map)

b_t041 = c_t041 = both_c = both_w = 0
for k in keys:
    doc = ds.get_document(k[0])
    ann = doc.annotations.get(k[1])
    gold = ann.label
    rc, ac = rag_pred_map[k] == gold, agent_pred_map[k] == gold
    if rc and ac: both_c += 1
    elif not rc and not ac: both_w += 1
    elif rc and not ac: b_t041 += 1  # RAG right, agent wrong = regression
    else: c_t041 += 1                 # RAG wrong, agent right = recovery

n_t041 = b_t041 + c_t041
result_t041 = stats.binomtest(min(b_t041, c_t041), n_t041, 0.5)

print(f"T041 hosted, FULL {len(keys)}-case test set (recomputed directly from the current saved "
      f"predictions, not the earlier 500-case subsample summary):")
print(f"  both_correct={both_c}  both_wrong={both_w}  "
      f"b(RAG-only-correct, regression)={b_t041}  c(agent-only-correct, recovery)={c_t041}")
print(f"  n_discordant={n_t041}, p={result_t041.pvalue:.4f}")
print()
if b_t041 > c_t041:
    print("*** REAL, CURRENT FINDING: at the full 2,091-case scale, regression now EXCEEDS "
          "recovery (the opposite direction from the 500-case subsample this project reported "
          "earlier - see docs/decisions.md ADR-007's caveat). The agent's accuracy at full scale "
          f"({agent_rec_accuracy:.1%}) is actually BELOW plain RAG's ({rag_rec_accuracy:.1%}). "
          "McNemar's test is still not significant (p={:.3f}), so this is not proven either - but "
          "the point estimate has flipped, not just weakened. ***".format(result_t041.pvalue))
else:
    print("Recovery still exceeds regression at full scale, consistent with the smaller sample.")


00:40:41 INFO     [ndatrace.parser] Parsing ContractNLI file: test.json


00:40:41 INFO     [ndatrace.parser] Parsed 123 documents from test (0 errors, 17 hypotheses)


T041 hosted, FULL 2091-case test set (recomputed directly from the current saved predictions, not the earlier 500-case subsample summary):
  both_correct=1559  both_wrong=380  b(RAG-only-correct, regression)=87  c(agent-only-correct, recovery)=65
  n_discordant=152, p=0.0882

*** REAL, CURRENT FINDING: at the full 2,091-case scale, regression now EXCEEDS recovery (the opposite direction from the 500-case subsample this project reported earlier - see docs/decisions.md ADR-007's caveat). The agent's accuracy at full scale (77.7%) is actually BELOW plain RAG's (78.7%). McNemar's test is still not significant (p=0.088), so this is not proven either - but the point estimate has flipped, not just weakened. ***


## 7. Conclusion

The selective agent improved development-set accuracy numerically (88.0% → 90.0% overall on the
150-case dev sample; 80.6% → 85.1% on the REVIEW-routed subset specifically), with recovery
beating regression 2-to-1 on that 67-case sample (McNemar p=0.51, not significant).

**A real, more complete finding emerged from recomputing this notebook against the current, full
2,091-case T041 hosted result files (2026-09-25) rather than relying on the earlier 500-case
subsample summary recorded in `docs/decisions.md`:** at full test-set scale, the direction
reverses. Regression (87 cases) now exceeds recovery (65 cases), and RAG+agent's overall accuracy
(77.7%) is actually *below* plain RAG's (78.7%). McNemar's test on this larger sample is still not
significant (p=0.088), so neither the earlier "recovery wins" nor this "regression wins" reading is
statistically proven — but the point estimate has flipped, not merely weakened, between the
500-case and full 2,091-case runs.

**This is disclosed here as an open, unresolved contradiction, not resolved in either direction.**
It directly complicates the architecture freeze's empirical basis (`docs/decisions.md` ADR-007,
ADR-009), which was made before this full-scale result existed and is not being retroactively
changed here (per the project's own "no re-tuning on test-set results" rule — the freeze predates
this data). What this notebook adds is an honest record that the full-scale hosted test-set result
does not confirm the dev-sample agent-inclusion rationale, and a professor/reviewer should weigh
this directly rather than see only the earlier, more favorable 500-case number. See
`docs/evaluation_protocol.md`'s "Current evaluation status" for how this interacts with the
still-unbackfilled joint-evidence metric on these same files.

A separate, real, disclosed limitation found in the same body of work: the agent's
`search_exceptions` tool did not reliably catch exception/carve-out clause patterns even when it
should have (`docs/decisions.md` ADR-011) — the agent's benefit, whichever direction it nets out to,
is uneven across failure types, not uniform.
